# Tarea 3 - Análisis de datos en tiempo real con Apache Kafka y Spark Streaming
Néstor Miguel Pacheco Ferrer - 1129502093

Ejecuta las celdas **una por una, en orden**, con el botón ▶ de cada celda. Las capturas para el Word se toman en las celdas marcadas con 📸.

## 1. Instalar Java, PySpark y la librería de Kafka para Python

In [ ]:
!apt-get update -qq > /dev/null
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!pip install -q pyspark==3.5.3 kafka-python
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!java -version
!spark-submit --version 2>&1 | grep -E 'version|Scala'

## 2. Descargar, descomprimir y mover Apache Kafka 3.9.2

In [ ]:
!wget -q https://downloads.apache.org/kafka/3.9.2/kafka_2.12-3.9.2.tgz || wget -q https://archive.apache.org/dist/kafka/3.9.2/kafka_2.12-3.9.2.tgz
!tar -xzf kafka_2.12-3.9.2.tgz
!rm -rf /opt/Kafka && mv kafka_2.12-3.9.2 /opt/Kafka
!ls /opt/Kafka

## 3. Iniciar ZooKeeper y el servidor de Kafka 📸 (Figura 6)

In [ ]:
!nohup /opt/Kafka/bin/zookeeper-server-start.sh /opt/Kafka/config/zookeeper.properties > zookeeper.log 2>&1 &
!sleep 10
!nohup /opt/Kafka/bin/kafka-server-start.sh /opt/Kafka/config/server.properties > kafka.log 2>&1 &
!sleep 15
!echo '--- ZooKeeper ---' && grep -i 'binding to port' zookeeper.log | tail -1
!echo '--- Kafka ---' && grep -i 'started (kafka.server' kafka.log | tail -1

## 4. Crear el topic sensor_data 📸 (Figura 7)

In [ ]:
!/opt/Kafka/bin/kafka-topics.sh --create --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1 --topic sensor_data
!/opt/Kafka/bin/kafka-topics.sh --describe --bootstrap-server localhost:9092 --topic sensor_data

## 5. Crear el productor kafka_producer.py

In [ ]:
%%writefile kafka_producer.py
import time
import json
import random
from kafka import KafkaProducer

def generate_sensor_data():
    return {
        "sensor_id": random.randint(1, 10),
        "temperature": round(random.uniform(20, 30), 2),
        "humidity": round(random.uniform(30, 70), 2),
        "timestamp": int(time.time())
    }

producer = KafkaProducer(bootstrap_servers=['localhost:9092'],
                         value_serializer=lambda x: json.dumps(x).encode('utf-8'))

while True:
    sensor_data = generate_sensor_data()
    producer.send('sensor_data', value=sensor_data)
    print(f"Sent: {sensor_data}")
    time.sleep(1)


## 6. Ejecutar el productor en segundo plano 📸 (Figura 8)

In [ ]:
!nohup python -u kafka_producer.py > producer.log 2>&1 &
!sleep 8
!tail -n 8 producer.log

## 7. Crear el consumidor de Spark Streaming del Anexo 3

In [ ]:
%%writefile spark_streaming_consumer.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, TimestampType
import logging

# Configura el nivel de log a WARN para reducir los mensajes INFO
spark = SparkSession.builder \
    .appName("KafkaSparkStreaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Definir el esquema de los datos de entrada
schema = StructType([
    StructField("sensor_id", IntegerType()),
    StructField("temperature", FloatType()),
    StructField("humidity", FloatType()),
    StructField("timestamp", TimestampType())
])

# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("SensorDataAnalysis") \
    .getOrCreate()

# Configurar el lector de streaming para leer desde Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "sensor_data") \
    .load()

# Parsear los datos JSON
parsed_df = df.select(from_json(col("value").cast("string"), schema).alias("data")).select("data.*")

# Calcular estadísticas por ventana de tiempo
windowed_stats = parsed_df \
    .groupBy(window(col("timestamp"), "1 minute"), "sensor_id") \
    .agg({"temperature": "avg", "humidity": "avg"})

# Escribir los resultados en la consola
query = windowed_stats \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()


## 8. Ejecutar el consumidor durante 2,5 minutos 📸 (Figura 9)
Espera a que aparezcan las tablas `Batch: 1`, `Batch: 2`... La celda se detiene sola al terminar el tiempo.

In [ ]:
!timeout 150 spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3 spark_streaming_consumer.py 2>/dev/null

## 9. Crear el consumidor ampliado y el script de la gráfica

In [ ]:
%%writefile spark_streaming_consumer_v2.py
# -*- coding: utf-8 -*-
"""
Consumidor ampliado de Spark Structured Streaming (Tarea 3 - Big Data UNAD)
Nestor Miguel Pacheco Ferrer - 1129502093

Lee el topic sensor_data de Kafka y por cada ventana de 1 minuto y sensor calcula:
numero de lecturas, temperatura promedio, maxima y minima, humedad promedio y
cuantas lecturas superan 28 grados (alerta). Cada micro-lote se muestra en consola
y se guarda en CSV para graficarlo despues con graficar_streaming.py.

Ejecucion:
spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3 spark_streaming_consumer_v2.py
"""
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, TimestampType

SALIDA = "resultados_streaming"
os.makedirs(SALIDA, exist_ok=True)
UMBRAL = 28.0

spark = SparkSession.builder.appName("SensorDataAnalysisV2").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("sensor_id", IntegerType()),
    StructField("temperature", FloatType()),
    StructField("humidity", FloatType()),
    StructField("timestamp", TimestampType())
])

# fuente: Kafka (o una carpeta con JSON si se pasa como argumento, solo para pruebas)
if len(sys.argv) > 1:
    parsed = spark.readStream.schema(schema).json(sys.argv[1])
else:
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", "localhost:9092")
           .option("subscribe", "sensor_data")
           .option("startingOffsets", "latest")
           .load())
    parsed = (raw.select(F.from_json(F.col("value").cast("string"), schema).alias("data"))
                 .select("data.*"))

stats = (parsed
         .withWatermark("timestamp", "2 minutes")
         .groupBy(F.window("timestamp", "1 minute"), "sensor_id")
         .agg(F.count("*").alias("lecturas"),
              F.round(F.avg("temperature"), 2).alias("temp_prom"),
              F.round(F.max("temperature"), 2).alias("temp_max"),
              F.round(F.min("temperature"), 2).alias("temp_min"),
              F.round(F.avg("humidity"), 2).alias("hum_prom"),
              F.sum(F.when(F.col("temperature") > UMBRAL, 1).otherwise(0)).alias("alertas"))
         .select(F.col("window.start").alias("inicio"), F.col("window.end").alias("fin"),
                 "sensor_id", "lecturas", "temp_prom", "temp_max", "temp_min",
                 "hum_prom", "alertas"))

def procesar_lote(lote, epoch_id):
    lote = lote.orderBy("inicio", "sensor_id")
    total = lote.agg(F.sum("lecturas").alias("t"), F.sum("alertas").alias("a")).collect()[0]
    print(f"\n===== Micro-lote {epoch_id} | lecturas: {total['t']} | alertas > {UMBRAL} C: {total['a']} =====")
    lote.show(30, truncate=False)
    lote.toPandas().to_csv(f"{SALIDA}/lote_{epoch_id:04d}.csv", index=False)

query = (stats.writeStream
         .outputMode("complete")
         .foreachBatch(procesar_lote)
         .trigger(processingTime="10 seconds")
         .start())
query.awaitTermination()


In [ ]:
%%writefile graficar_streaming.py
# Grafica el ultimo micro-lote guardado por spark_streaming_consumer_v2.py
import glob
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ultimo = sorted(glob.glob("resultados_streaming/lote_*.csv"))[-1]
df = pd.read_csv(ultimo, parse_dates=["inicio"])
print("Archivo:", ultimo, "| filas:", len(df))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for sensor, g in df.groupby("sensor_id"):
    axes[0].plot(g["inicio"].dt.strftime("%H:%M"), g["temp_prom"], marker="o", label=f"Sensor {sensor}")
axes[0].set_title("Temperatura promedio por ventana de 1 minuto")
axes[0].set_xlabel("Inicio de la ventana"); axes[0].set_ylabel("°C")
axes[0].legend(fontsize=7, ncol=2)

tot = df.groupby("sensor_id")[["lecturas", "alertas"]].sum()
axes[1].bar(tot.index.astype(str), tot["lecturas"], label="Lecturas", color="#4C72B0")
axes[1].bar(tot.index.astype(str), tot["alertas"], label="Alertas > 28 °C", color="#C44E52")
axes[1].set_title("Lecturas y alertas acumuladas por sensor")
axes[1].set_xlabel("Sensor"); axes[1].legend()
plt.tight_layout(); plt.savefig("fig_streaming.png", dpi=200)
print("Grafica guardada en fig_streaming.png")


## 10. Ejecutar el consumidor ampliado en segundo plano

In [ ]:
!rm -rf resultados_streaming
!nohup spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3 spark_streaming_consumer_v2.py > consumer_v2.log 2>/dev/null &
!sleep 90
!tail -n 40 consumer_v2.log

## 11. Interfaz web de Spark 📸 (Figura 11)
Colab no deja abrir el puerto 4040 en el navegador, así que se consultan los jobs, stages y executors con la API REST de la misma interfaz. Captura las tablas.

In [ ]:
import time, requests, pandas as pd
from IPython.display import display
# el consumidor ampliado de la celda 10 sigue corriendo; se consulta la API REST de su interfaz web
app = None
for i in range(24):
    for puerto in (4040, 4041, 4042):
        try:
            base = f"http://localhost:{puerto}/api/v1"
            app = requests.get(f"{base}/applications", timeout=3).json()[0]
            break
        except Exception:
            pass
    if app: break
    time.sleep(5)
print("Interfaz web de Spark en el puerto", puerto)
print("Aplicacion:", app["name"], "| ID:", app["id"])
jobs = pd.DataFrame(requests.get(f"{base}/applications/{app['id']}/jobs").json())
print("\nJOBS")
display(jobs[["jobId", "status", "submissionTime", "completionTime", "numTasks", "numCompletedTasks", "numCompletedStages"]].head(15))
stages = pd.DataFrame(requests.get(f"{base}/applications/{app['id']}/stages").json())
print("\nSTAGES")
display(stages[["stageId", "status", "numTasks", "inputRecords", "executorRunTime", "name"]].head(10))
ex = pd.DataFrame(requests.get(f"{base}/applications/{app['id']}/executors").json())
print("\nEXECUTORS")
display(ex[["id", "hostPort", "totalCores", "maxMemory", "completedTasks", "totalDuration"]])

## 12. Resultado del consumidor ampliado y gráfica 📸 (Figura 10)

In [ ]:
!sleep 60
!tail -n 40 consumer_v2.log
!python graficar_streaming.py
from IPython.display import Image, display
display(Image('fig_streaming.png', width=900))

## 13. Detener todo

In [ ]:
!pkill -f spark_streaming_consumer_v2.py
!pkill -f kafka_producer.py
!/opt/Kafka/bin/kafka-server-stop.sh
!sleep 5
!/opt/Kafka/bin/zookeeper-server-stop.sh
print('Procesos detenidos')